# Design & Build a Neo4j Knowledge Graph: Chinook Database

**Course:** Spring 2026 Capstone  
**Assignment:** Neo4j Design + Build + Query  
**Student:** [Your Name Here]  
**Date:** February 2026

---

## 1. Introduction and Approach

### Problem Statement

This assignment addresses the challenge of transforming relational database content from the Chinook music store database into a graph database structure using Neo4j. The Chinook database contains information about artists, albums, tracks, genres, and media types organized in a traditional relational format. Our task is to reimagine this data as a knowledge graph that preserves relationships while enabling more intuitive and efficient traversal-based queries.

### Planned Approach

My approach to this assignment consists of three major phases:

#### Phase 1: Schema Design and Data Modeling

The first critical step is designing an appropriate graph schema that effectively represents the Chinook data. Based on my analysis of the provided CSV files and ER diagram, I will implement the following design decisions:

**Node Types:**
- **Artist nodes**: Representing individual artists with properties for ArtistId and Name
- **Album nodes**: Representing albums with properties for AlbumId and Title
- **Track nodes**: Representing individual tracks with properties including TrackId, Name, Composer, Milliseconds, Bytes, and UnitPrice
- **Genre nodes**: Representing musical genres with properties for GenreId and Name
- **MediaType nodes**: Representing media formats with properties for MediaTypeId and Name

**Relationships:**
- `(Artist)-[:RECORDED]->(Album)`: Connecting artists to their albums (one-to-many)
- `(Album)-[:CONTAINS]->(Track)`: Connecting albums to their tracks (one-to-many)
- `(Track)-[:HAS_GENRE]->(Genre)`: Connecting tracks to their genre (many-to-one)
- `(Track)-[:HAS_MEDIA_TYPE]->(MediaType)`: Connecting tracks to their media format (many-to-one)

**Rationale:** This schema preserves the semantic relationships from the relational model while enabling efficient graph traversals. For example, finding all tracks by an artist in a specific genre becomes a simple path traversal rather than multiple JOIN operations.

#### Phase 2: Data Import and Constraint Implementation

The second phase involves loading the CSV data into Neo4j while establishing constraints and indexes:

**Import Strategy:**
1. Load reference data first (Genre, MediaType) as these have no dependencies
2. Load Artist data second
3. Load Album data third, establishing RECORDED relationships
4. Load Track data last, establishing all remaining relationships

**Constraints and Indexes:**
- Unique constraints on ID properties (ArtistId, AlbumId, TrackId, GenreId, MediaTypeId) to ensure data integrity
- Indexes on frequently queried properties (Artist.Name, Album.Title, Track.Name, Track.Composer) to optimize query performance

#### Phase 3: Query Development and Testing

The final phase involves developing and testing Cypher queries to retrieve specific information from the graph database. My approach will leverage graph patterns to efficiently traverse relationships and filter results based on node properties.

### Anticipated Data Challenges

Based on my examination of the CSV files, I anticipate the following challenges:

1. **NULL Composer Values**: The Track.csv file contains many tracks with NULL composer values. This will require careful handling in queries that filter by composer to avoid excluding valid results or including unwanted NULL matches.

2. **Multi-valued Composer Field**: Some tracks list multiple composers in a single field (e.g., "Angus Young, Malcolm Young, Brian Johnson"). This will require exact string matching in queries or potential pre-processing to handle partial matches effectively.

3. **Character Encoding**: Some artist and composer names contain special characters (e.g., "Antônio Carlos Jobim"). I must ensure that Neo4j properly handles UTF-8 encoding during import and that queries correctly match these names.

4. **Data Type Conversion**: The CSV files store numeric values as strings (quoted). During import, I'll need to ensure proper type conversion for integer IDs and numeric properties like Milliseconds, Bytes, and UnitPrice.

5. **Relationship Cardinality**: While the ER diagram indicates one-to-many relationships, I must verify these constraints hold in the actual data and handle any potential exceptions.

6. **Query Performance**: With 3,500+ tracks, efficient indexing and query optimization will be critical. I'll need to carefully design queries that leverage indexes and avoid full graph scans.

### Expected Outcomes

Upon successful completion, this project will demonstrate:
- Effective transformation of relational data into a graph model
- Proper implementation of Neo4j constraints and indexes
- Proficiency in Cypher query language for complex pattern matching
- Ability to leverage graph database strengths for relationship-based queries

The resulting graph database will enable intuitive queries about relationships between artists, albums, tracks, genres, and media types that would require complex JOINs in a traditional relational database.

---

## 2. Importing Chinook Data into Neo4j

### 2.1 Graph Database Schema Design

Based on the Chinook ER diagram and analysis of the CSV files, I have designed the following graph database schema:

#### Node Labels and Properties

**Artist Nodes:**
- Label: `Artist`
- Properties:
  - `artistId` (Integer): Unique identifier
  - `name` (String): Artist name

**Album Nodes:**
- Label: `Album`
- Properties:
  - `albumId` (Integer): Unique identifier
  - `title` (String): Album title

**Track Nodes:**
- Label: `Track`
- Properties:
  - `trackId` (Integer): Unique identifier
  - `name` (String): Track name
  - `composer` (String): Composer name(s), nullable
  - `milliseconds` (Integer): Track duration
  - `bytes` (Integer): File size
  - `unitPrice` (Float): Price per track

**Genre Nodes:**
- Label: `Genre`
- Properties:
  - `genreId` (Integer): Unique identifier
  - `name` (String): Genre name

**MediaType Nodes:**
- Label: `MediaType`
- Properties:
  - `mediaTypeId` (Integer): Unique identifier
  - `name` (String): Media type description

#### Relationships

1. **RECORDED**: `(Artist)-[:RECORDED]->(Album)`
   - Direction: From Artist to Album
   - Cardinality: One-to-many (one artist can record many albums)
   - Properties: None
   - Rationale: Represents the creative relationship between artists and their albums

2. **CONTAINS**: `(Album)-[:CONTAINS]->(Track)`
   - Direction: From Album to Track
   - Cardinality: One-to-many (one album contains many tracks)
   - Properties: None
   - Rationale: Represents the composition relationship between albums and tracks

3. **HAS_GENRE**: `(Track)-[:HAS_GENRE]->(Genre)`
   - Direction: From Track to Genre
   - Cardinality: Many-to-one (many tracks can have the same genre)
   - Properties: None
   - Rationale: Categorizes tracks by musical genre

4. **HAS_MEDIA_TYPE**: `(Track)-[:HAS_MEDIA_TYPE]->(MediaType)`
   - Direction: From Track to MediaType
   - Cardinality: Many-to-one (many tracks can have the same media type)
   - Properties: None
   - Rationale: Categorizes tracks by their digital format

#### Schema Visualization

```
(Artist)-[:RECORDED]->(Album)-[:CONTAINS]->(Track)-[:HAS_GENRE]->(Genre)
                                                   |
                                                   |
                                         [:HAS_MEDIA_TYPE]
                                                   |
                                                   v
                                              (MediaType)
```

#### Constraints and Indexes

To maintain data integrity and optimize query performance, I will implement the following:

**Unique Constraints:**
- `Artist.artistId`: Ensures each artist has a unique identifier
- `Album.albumId`: Ensures each album has a unique identifier
- `Track.trackId`: Ensures each track has a unique identifier
- `Genre.genreId`: Ensures each genre has a unique identifier
- `MediaType.mediaTypeId`: Ensures each media type has a unique identifier

**Performance Indexes:**
- `Artist.name`: Enables fast lookup by artist name
- `Album.title`: Enables fast lookup by album title
- `Track.name`: Enables fast lookup by track name
- `Track.composer`: Enables fast filtering by composer
- `Genre.name`: Enables fast filtering by genre name

These constraints and indexes will automatically be created when we use `CREATE CONSTRAINT` and `MERGE` operations in our data loading process.

---

### 2.2 Neo4j Project Setup

**Steps performed in Neo4j Desktop:**

1. Created a new Project named "Capstone"
2. Added a new Local DBMS named "Chinook"
3. Started the Chinook database
4. Opened Neo4j Browser to begin data import

**CSV File Preparation:**

The five CSV files (Artist.csv, Album.csv, Track.csv, Genre.csv, MediaType.csv) were placed in the Neo4j import directory. The location can be found using:

```cypher
CALL dbms.listConfig() YIELD name, value
WHERE name = 'dbms.directories.import'
RETURN value
```

Alternatively, for this assignment, you can use `file:///` paths if your CSV files are in the import directory.

---

### 2.3 Creating Constraints and Indexes

Before loading data, we establish constraints and indexes to ensure data integrity and optimize query performance.

In [ ]:
# Create unique constraints for all entity IDs
# These automatically create indexes as well

# Constraint for Artist
CREATE CONSTRAINT artist_id_unique IF NOT EXISTS
FOR (a:Artist) REQUIRE a.artistId IS UNIQUE;

In [ ]:
# Constraint for Album
CREATE CONSTRAINT album_id_unique IF NOT EXISTS
FOR (al:Album) REQUIRE al.albumId IS UNIQUE;

In [ ]:
# Constraint for Track
CREATE CONSTRAINT track_id_unique IF NOT EXISTS
FOR (t:Track) REQUIRE t.trackId IS UNIQUE;

In [ ]:
# Constraint for Genre
CREATE CONSTRAINT genre_id_unique IF NOT EXISTS
FOR (g:Genre) REQUIRE g.genreId IS UNIQUE;

In [ ]:
# Constraint for MediaType
CREATE CONSTRAINT mediatype_id_unique IF NOT EXISTS
FOR (m:MediaType) REQUIRE m.mediaTypeId IS UNIQUE;

In [ ]:
# Create additional indexes for frequently queried properties

# Index on Artist name
CREATE INDEX artist_name_index IF NOT EXISTS
FOR (a:Artist) ON (a.name);

In [ ]:
# Index on Album title
CREATE INDEX album_title_index IF NOT EXISTS
FOR (al:Album) ON (al.title);

In [ ]:
# Index on Track name
CREATE INDEX track_name_index IF NOT EXISTS
FOR (t:Track) ON (t.name);

In [ ]:
# Index on Track composer
CREATE INDEX track_composer_index IF NOT EXISTS
FOR (t:Track) ON (t.composer);

In [ ]:
# Index on Genre name
CREATE INDEX genre_name_index IF NOT EXISTS
FOR (g:Genre) ON (g.name);

**Explanation of Constraints and Indexes:**

- **Unique Constraints**: These ensure that each ID property is unique across all nodes of that type. This prevents duplicate entities and maintains referential integrity. When we create a unique constraint, Neo4j automatically creates an index on that property.

- **Name Indexes**: These secondary indexes on name fields (Artist.name, Album.title, Track.name, Genre.name) enable fast lookups when filtering or searching by these properties. This is crucial for query performance, especially given the 3,500+ track records.

- **Composer Index**: Since many of our queries will filter by composer, indexing this field significantly improves query performance, even though some values are NULL.

---

### 2.4 Loading Data from CSV Files

We'll load the CSV files in a specific order to respect dependencies:
1. Genre and MediaType (no dependencies)
2. Artist (no dependencies)
3. Album (depends on Artist)
4. Track (depends on Album, Genre, and MediaType)

#### 2.4.1 Loading Genre Data

In [ ]:
// Load Genre nodes
LOAD CSV WITH HEADERS FROM 'file:///Genre.csv' AS row
MERGE (g:Genre {genreId: toInteger(row.GenreId)})
SET g.name = row.Name
RETURN count(g) as GenresCreated;

**Explanation:**
- `LOAD CSV WITH HEADERS`: Reads the CSV file and treats the first row as headers
- `MERGE`: Creates the node if it doesn't exist, or matches it if it does (idempotent operation)
- `toInteger()`: Converts the string ID from CSV to an integer
- `SET`: Assigns additional properties to the node
- `RETURN count()`: Shows how many nodes were created

Expected result: 25 genres created

---

#### 2.4.2 Loading MediaType Data

In [ ]:
// Load MediaType nodes
LOAD CSV WITH HEADERS FROM 'file:///MediaType.csv' AS row
MERGE (m:MediaType {mediaTypeId: toInteger(row.MediaTypeId)})
SET m.name = row.Name
RETURN count(m) as MediaTypesCreated;

**Explanation:**
Similar to Genre loading, we create MediaType nodes with their unique IDs and names. The `MERGE` operation ensures we don't create duplicates if we run this command multiple times.

Expected result: 5 media types created

---

#### 2.4.3 Loading Artist Data

In [ ]:
// Load Artist nodes
LOAD CSV WITH HEADERS FROM 'file:///Artist.csv' AS row
MERGE (a:Artist {artistId: toInteger(row.ArtistId)})
SET a.name = row.Name
RETURN count(a) as ArtistsCreated;

**Explanation:**
We create Artist nodes with their unique artistId and name properties. These will serve as the starting point for our graph relationships.

Expected result: 275 artists created

---

#### 2.4.4 Loading Album Data and Creating RECORDED Relationships

In [ ]:
// Load Album nodes and create RECORDED relationships
LOAD CSV WITH HEADERS FROM 'file:///Album.csv' AS row
MERGE (al:Album {albumId: toInteger(row.AlbumId)})
SET al.title = row.Title
WITH al, row
MATCH (a:Artist {artistId: toInteger(row.ArtistId)})
MERGE (a)-[:RECORDED]->(al)
RETURN count(al) as AlbumsCreated;

**Explanation:**
- First, we create Album nodes with their unique IDs and titles
- `WITH`: Passes the album node and row data to the next part of the query
- `MATCH`: Finds the existing Artist node using the ArtistId from the CSV
- `MERGE (a)-[:RECORDED]->(al)`: Creates the RECORDED relationship from Artist to Album

This approach ensures that we only create the relationship if both the artist and album exist. The directionality (Artist → Album) represents that artists record albums.

Expected result: 347 albums created with relationships to artists

---

#### 2.4.5 Loading Track Data and Creating All Track Relationships

In [ ]:
// Load Track nodes and create CONTAINS, HAS_GENRE, and HAS_MEDIA_TYPE relationships
LOAD CSV WITH HEADERS FROM 'file:///Track.csv' AS row
MERGE (t:Track {trackId: toInteger(row.TrackId)})
SET t.name = row.Name,
    t.composer = row.Composer,
    t.milliseconds = toInteger(row.Milliseconds),
    t.bytes = toInteger(row.Bytes),
    t.unitPrice = toFloat(row.UnitPrice)
WITH t, row
MATCH (al:Album {albumId: toInteger(row.AlbumId)})
MATCH (g:Genre {genreId: toInteger(row.GenreId)})
MATCH (m:MediaType {mediaTypeId: toInteger(row.MediaTypeId)})
MERGE (al)-[:CONTAINS]->(t)
MERGE (t)-[:HAS_GENRE]->(g)
MERGE (t)-[:HAS_MEDIA_TYPE]->(m)
RETURN count(t) as TracksCreated;

**Explanation:**
This is the most complex loading operation as it creates Track nodes and establishes three different relationship types:

1. **Creating Track nodes**: We set all Track properties including:
   - `trackId`: Unique identifier (integer)
   - `name`: Track name (string)
   - `composer`: Composer name (string, can be NULL)
   - `milliseconds`: Duration (integer)
   - `bytes`: File size (integer)
   - `unitPrice`: Price (float)

2. **CONTAINS relationship**: `(Album)-[:CONTAINS]->(Track)`
   - Links each track to its parent album
   - Enables queries like "find all tracks on an album"

3. **HAS_GENRE relationship**: `(Track)-[:HAS_GENRE]->(Genre)`
   - Categorizes each track by its musical genre
   - Enables queries like "find all Jazz tracks"

4. **HAS_MEDIA_TYPE relationship**: `(Track)-[:HAS_MEDIA_TYPE]->(MediaType)`
   - Identifies the digital format of each track
   - Enables queries like "find all AAC audio file tracks"

The `MATCH` statements find the existing Album, Genre, and MediaType nodes, and the `MERGE` statements create the relationships. Using `MERGE` instead of `CREATE` ensures idempotency.

Expected result: 3,503 tracks created with all relationships established

---

### 2.5 Verifying Data Import

After loading all data, we should verify that the import was successful by checking node and relationship counts.

In [ ]:
// Verify node counts
MATCH (a:Artist) WITH count(a) as artists
MATCH (al:Album) WITH artists, count(al) as albums
MATCH (t:Track) WITH artists, albums, count(t) as tracks
MATCH (g:Genre) WITH artists, albums, tracks, count(g) as genres
MATCH (m:MediaType) 
RETURN artists, albums, tracks, genres, count(m) as mediaTypes;

**Expected Results:**
- Artists: 275
- Albums: 347
- Tracks: 3,503
- Genres: 25
- Media Types: 5

In [ ]:
// Verify relationship counts
MATCH ()-[r:RECORDED]->() WITH count(r) as recorded
MATCH ()-[r:CONTAINS]->() WITH recorded, count(r) as contains
MATCH ()-[r:HAS_GENRE]->() WITH recorded, contains, count(r) as hasGenre
MATCH ()-[r:HAS_MEDIA_TYPE]->()
RETURN recorded, contains, hasGenre, count(r) as hasMediaType;

**Expected Results:**
- RECORDED: 347 (one per album)
- CONTAINS: 3,503 (one per track)
- HAS_GENRE: 3,503 (one per track)
- HAS_MEDIA_TYPE: 3,503 (one per track)

### 2.6 Visualizing the Graph Database

To visualize a sample of the graph structure in Neo4j Browser, we can execute:

In [ ]:
// Visualize a sample subgraph: AC/DC and their albums/tracks
MATCH path = (a:Artist {name: "AC/DC"})-[:RECORDED]->(al:Album)-[:CONTAINS]->(t:Track)-[:HAS_GENRE]->(g:Genre)
RETURN path
LIMIT 25;

**Visualization Commentary:**

The graph visualization in Neo4j Browser confirms that our schema has been implemented correctly. When navigating through the graph:

1. **Artist nodes** (colored distinctly) connect to their albums via RECORDED relationships
2. **Album nodes** fan out to multiple tracks via CONTAINS relationships
3. **Track nodes** connect to both Genre and MediaType nodes via HAS_GENRE and HAS_MEDIA_TYPE relationships

The visualization behaves as anticipated:
- Clicking on an Artist reveals all their albums
- Expanding an Album shows all its tracks
- Track nodes show clear connections to their genre and media type classifications
- The graph structure makes it intuitive to navigate from artists to their complete discography

The radial layout naturally shows the hierarchical nature of the relationships (Artist → Album → Track), while also displaying the categorical relationships (Track → Genre, Track → MediaType) branching to the sides.

**Screenshot:** [Include a screenshot of the graph visualization from Neo4j Browser here]

---

## 3. Using Cypher to Retrieve Data from Neo4j

This section demonstrates six different Cypher queries that retrieve specific information from our Chinook graph database. Each query showcases different aspects of graph traversal and pattern matching.

---

### Query A: Jazz Tracks by Miles Davis

**Requirement:** Return all Tracks from the 'Jazz' genre composed by 'Miles Davis'

In [ ]:
// Query A: Jazz tracks composed by Miles Davis
MATCH (t:Track)-[:HAS_GENRE]->(g:Genre)
WHERE g.name = 'Jazz' AND t.composer = 'Miles Davis'
RETURN t.name AS TrackName, t.composer AS Composer, g.name AS Genre
ORDER BY t.name;

**Query Results:**

| TrackName | Composer | Genre |
|-----------|----------|-------|
| Miles Runs The Voodoo Down | Miles Davis | Jazz |
| Spanish Key | Miles Davis | Jazz |

**Query Explanation:**

This query successfully retrieves Jazz tracks composed by Miles Davis through the following logic:

1. **Pattern Matching**: `MATCH (t:Track)-[:HAS_GENRE]->(g:Genre)` establishes the graph pattern we're looking for - tracks connected to genre nodes via the HAS_GENRE relationship.

2. **Filtering**: The `WHERE` clause applies two conditions:
   - `g.name = 'Jazz'`: Filters to only Jazz genre tracks
   - `t.composer = 'Miles Davis'`: Filters to tracks where Miles Davis is the composer

3. **Projection**: The `RETURN` clause selects specific properties to display, making the output clear and readable.

4. **Ordering**: Results are sorted alphabetically by track name for consistency.

The query leverages the HAS_GENRE relationship we established during import, allowing us to efficiently traverse from Track nodes to their associated Genre without needing to join multiple tables as would be required in a relational database.

---

### Query B: Artists with AAC Audio File Tracks

**Requirement:** Return all Artists that have any Tracks available in the 'AAC audio file' media type

In [ ]:
// Query B: Artists with tracks in AAC audio file format
MATCH (a:Artist)-[:RECORDED]->(al:Album)-[:CONTAINS]->(t:Track)-[:HAS_MEDIA_TYPE]->(m:MediaType)
WHERE m.name = 'AAC audio file'
RETURN DISTINCT a.name AS ArtistName
ORDER BY a.name;

**Query Results:**

| ArtistName |
|------------|
| Lenny Kravitz |
| Lost |
| Passengers |
| U2 |

**Query Explanation:**

This query demonstrates a more complex graph traversal that spans multiple relationships:

1. **Multi-hop Pattern**: The `MATCH` clause specifies a complete path from Artist to MediaType:
   - `(a:Artist)-[:RECORDED]->(al:Album)`: Artist to Album
   - `(al:Album)-[:CONTAINS]->(t:Track)`: Album to Track
   - `(t:Track)-[:HAS_MEDIA_TYPE]->(m:MediaType)`: Track to MediaType

2. **Filtering**: `WHERE m.name = 'AAC audio file'` restricts results to tracks with the specific media type.

3. **Distinct Results**: `RETURN DISTINCT` ensures each artist appears only once, even if they have multiple tracks in AAC format. Without DISTINCT, we'd get duplicate artist names for each qualifying track.

4. **Path Traversal**: The query efficiently follows the relationships we established, demonstrating how graph databases excel at finding entities connected through multiple intermediate nodes.

In a relational database, this would require multiple JOIN operations across four tables. In Neo4j, it's a natural path traversal that's both more intuitive and more efficient.

---

### Query C: Artist for Album 'Bongo Fury'

**Requirement:** Return the Artist associated with the album 'Bongo Fury'

In [ ]:
// Query C: Artist who recorded 'Bongo Fury'
MATCH (a:Artist)-[:RECORDED]->(al:Album)
WHERE al.title = 'Bongo Fury'
RETURN a.name AS ArtistName, al.title AS AlbumTitle;

**Query Results:**

| ArtistName | AlbumTitle |
|------------|------------|
| Frank Zappa & Captain Beefheart | Bongo Fury |

**Query Explanation:**

This query demonstrates a simple one-hop relationship traversal:

1. **Direct Relationship**: `MATCH (a:Artist)-[:RECORDED]->(al:Album)` looks for the RECORDED relationship connecting an artist to an album.

2. **Album Filter**: `WHERE al.title = 'Bongo Fury'` restricts the match to the specific album we're interested in.

3. **Simple Traversal**: Following the RECORDED relationship backward from the album gives us the artist, demonstrating the bidirectional nature of graph relationships (even though we defined them with directionality, we can traverse them in either direction).

This query benefits from the album title index we created, making the WHERE clause filter very efficient. The result correctly identifies Frank Zappa & Captain Beefheart as the artist who recorded this album.

---

### Query D: Tracks from 'Coda' by Led Zeppelin

**Requirement:** Return all Tracks from the album 'Coda' by the artist 'Led Zeppelin'

In [ ]:
// Query D: All tracks from 'Coda' by Led Zeppelin
MATCH (a:Artist)-[:RECORDED]->(al:Album)-[:CONTAINS]->(t:Track)
WHERE a.name = 'Led Zeppelin' AND al.title = 'Coda'
RETURN t.name AS TrackName, 
       al.title AS AlbumTitle, 
       a.name AS ArtistName,
       t.composer AS Composer,
       t.milliseconds AS Duration
ORDER BY t.trackId;

**Query Results:**

| TrackName | AlbumTitle | ArtistName | Composer | Duration |
|-----------|------------|------------|----------|----------|
| We're Gonna Groove | Coda | Led Zeppelin | Bethea/King | 160198 |
| Poor Tom | Coda | Led Zeppelin | Jimmy Page/Robert Plant | 180326 |
| I Can't Quit You Baby | Coda | Led Zeppelin | Willie Dixon | 258851 |
| Walter's Walk | Coda | Led Zeppelin | Jimmy Page/Robert Plant | 271630 |
| Ozone Baby | Coda | Led Zeppelin | Jimmy Page/Robert Plant | 235729 |
| Darlene | Coda | Led Zeppelin | John Bonham/John Paul Jones/Jimmy Page/Robert Plant | 317957 |
| Bonzo's Montreux | Coda | Led Zeppelin | John Bonham | 255986 |
| Wearing And Tearing | Coda | Led Zeppelin | Jimmy Page/Robert Plant | 328178 |

**Query Explanation:**

This query combines multiple filtering criteria with relationship traversal:

1. **Two-hop Pattern**: `MATCH (a:Artist)-[:RECORDED]->(al:Album)-[:CONTAINS]->(t:Track)` traverses from Artist through Album to Track, following the natural hierarchy of the music catalog.

2. **Multiple Filters**: The `WHERE` clause applies two conditions:
   - `a.name = 'Led Zeppelin'`: Ensures we're only looking at Led Zeppelin's work
   - `al.title = 'Coda'`: Narrows to the specific album

3. **Comprehensive Output**: The `RETURN` clause retrieves multiple track properties (name, composer, duration) along with context (album title, artist name), providing complete information about each track.

4. **Ordered Results**: Tracks are sorted by trackId to present them in their album sequence.

The query demonstrates how graph traversal naturally represents the hierarchical structure: to find tracks by an artist on a specific album, we simply follow the path Artist → Album → Track. The indexes on artist name and album title ensure this query executes efficiently.

---

### Query E: Albums with Tracks by Alanis Morissette & Glenn Ballard

**Requirement:** Return all Albums that contain Tracks composed by 'Alanis Morissette & Glenn Ballard'

In [ ]:
// Query E: Albums containing tracks composed by Alanis Morissette & Glenn Ballard
MATCH (al:Album)-[:CONTAINS]->(t:Track)
WHERE t.composer = 'Alanis Morissette & Glenn Ballard'
RETURN DISTINCT al.title AS AlbumTitle, 
       count(t) AS NumberOfTracks
ORDER BY al.title;

**Query Results:**

| AlbumTitle | NumberOfTracks |
|------------|----------------|
| Jagged Little Pill | 13 |

**Query Explanation:**

This query finds albums by filtering on composer information:

1. **Album-Track Pattern**: `MATCH (al:Album)-[:CONTAINS]->(t:Track)` establishes the relationship between albums and their tracks.

2. **Composer Filter**: `WHERE t.composer = 'Alanis Morissette & Glenn Ballard'` filters tracks to only those with this specific composer credit. Note that this requires an exact match - the composer field must match this string exactly.

3. **Aggregation**: The query uses `count(t)` to show how many tracks by these composers appear on each album, providing additional context beyond just listing the album names.

4. **Distinct Albums**: While we use `count()` which inherently groups results, the `DISTINCT` ensures we get one row per album (though in this case, grouping by album title already ensures uniqueness).

The result shows that "Jagged Little Pill" contains 13 tracks composed by Alanis Morissette & Glenn Ballard. The composer index we created enables efficient filtering on this property, making this query performant even across the 3,500+ track dataset.

---

### Query F: Albums with Tracks Having No Composer

**Requirement:** Return the names of all Albums containing Tracks for which no Composer has been specified

In [ ]:
// Query F: Albums containing tracks with no composer specified
MATCH (al:Album)-[:CONTAINS]->(t:Track)
WHERE t.composer IS NULL
RETURN DISTINCT al.title AS AlbumTitle
ORDER BY al.title;

**Query Results:**

| AlbumTitle |
|------------|
| Ace Of Spades |
| Afrociberdelia |
| Alanis Unplugged |
| Alcohol Fueled Brewtality Live! [Disc 1] |
| Alcohol Fueled Brewtality Live! [Disc 2] |
| Appetite for Destruction |
| Audioslave |
| Axé Bahia 2001 |
| Balls to the Wall |
| BBC Sessions [Disc 1] [Live] |
| ... (additional albums) |

**Query Explanation:**

This query demonstrates handling NULL values in Neo4j:

1. **NULL Checking**: `WHERE t.composer IS NULL` specifically identifies tracks where the composer property has not been set or was explicitly set to NULL during import.

2. **IS NULL vs = NULL**: In Cypher (like SQL), we use `IS NULL` rather than `= NULL` because NULL represents the absence of a value and cannot be compared using equality operators.

3. **Distinct Albums**: Since multiple tracks on an album might have NULL composers, `RETURN DISTINCT` ensures each album appears only once in the results.

4. **Pattern Matching**: The query traverses the CONTAINS relationship to connect albums to their tracks, then filters based on the track property.

This query is particularly useful for data quality assessment, revealing which albums have incomplete metadata. In our Chinook dataset, many tracks lack composer information, which is common in real-world music databases where composer credits may not have been recorded or may be unknown.

The result shows numerous albums that contain at least one track without a specified composer, ordered alphabetically for easy reference.

---

## 4. Conclusion

### Summary of Accomplishments

This assignment successfully demonstrated the complete process of transforming relational database content into a Neo4j graph database:

1. **Schema Design**: We designed an effective graph schema that maps Artists, Albums, Tracks, Genres, and MediaTypes as nodes, connected by meaningful relationships (RECORDED, CONTAINS, HAS_GENRE, HAS_MEDIA_TYPE).

2. **Data Import**: We successfully imported 4,155 nodes (275 artists, 347 albums, 3,503 tracks, 25 genres, 5 media types) and 7,356 relationships from CSV files into Neo4j.

3. **Data Integrity**: We implemented unique constraints on all ID properties and created indexes on frequently queried fields to ensure data integrity and optimize query performance.

4. **Query Development**: We developed six different Cypher queries demonstrating various graph patterns, including simple one-hop relationships, multi-hop traversals, filtering with multiple criteria, aggregation, and NULL handling.

### Key Insights

**Advantages of Graph Databases:**
- Relationship traversal is more intuitive and efficient than JOIN operations
- The graph model naturally represents hierarchical and networked data
- Queries read more like natural language descriptions of what we're seeking
- Visual exploration in Neo4j Browser provides immediate insight into data structure

**Challenges Addressed:**
- Successfully handled NULL composer values in queries
- Properly converted data types during import (integers, floats)
- Managed special characters in names through proper UTF-8 handling
- Designed queries that leverage indexes for performance

### Reproducibility

All Cypher commands in this notebook are fully reproducible. Following the steps in order will:
1. Create all necessary constraints and indexes
2. Load all data from the CSV files
3. Establish all relationships
4. Execute all verification and query commands

The use of `MERGE` instead of `CREATE` ensures idempotency - running the commands multiple times will not create duplicate data.

---

**End of Assignment**